 # Practice 9: PyTorch Lightning & Sentence Transformers

 **Session Goals:**
 1.  **Understand the "Why":** Why do we need PyTorch Lightning? What problems does it solve?
 2.  **Core Concepts:** Learn the two main components: `LightningModule` and `Trainer`.
 3.  **Practical Example:** Build a sentence similarity regressor using a pre-trained `sentence-transformers` model.

 ---

 ## Part 1: What is PyTorch Lightning?

 **Problem:** Standard PyTorch is incredibly flexible, but it requires a *lot* of boilerplate code.

 For any serious project, you have to manually write:
 * The training loop
 * The validation loop
 * `model.train()` and `model.eval()` switches
 * `optimizer.zero_grad()`
 * `loss.backward()`
 * `optimizer.step()`
 * Moving data to the GPU (`.to(device)`)
 * Logging metrics (e.g., to TensorBoard or a CSV)
 * Saving model checkpoints
 * ...and it gets *much* harder with multi-GPU training, 16-bit precision, etc.

 **Solution:** **PyTorch Lightning (PL)** is a lightweight wrapper that abstracts this away.

 > ⚡ **Analogy:** If PyTorch gives you the "engine," "wheels," and "steering wheel," PyTorch Lightning gives you the "car." You just need to define what happens inside the engine (`LightningModule`) and tell it when to drive (`Trainer`).

 **The Two Core Components:**
 1.  **`LightningModule`:** This is your model. It's just like a standard `nn.Module`, but it organizes your code into specific methods:
     * `__init__()`: Define your layers (e.g., `nn.Linear`, `SentenceTransformer`).
     * `forward()`: The "inference" pass.
     * `training_step()`: Defines what happens for *one batch* of training data (compute loss, log metrics).
     * `validation_step()`: Defines what happens for *one batch* of validation data.
     * `configure_optimizers()`: Tell PL which optimizer to use (e.g., Adam).

 2.  **`Trainer`:** This is the "driver." It handles all the boilerplate:
     * Runs the training and validation loops.
     * Automatically calls `.train()`, `.eval()`, `.zero_grad()`, `.backward()`, `.step()`.
     * Handles device placement (CPU/GPU/TPU).
     * Manages logging, checkpointing, and more.

 Today, we'll use PL to fine-tune a simple regression model on top of a powerful `sentence-transformers` model.


 ## Part 1.2: What is Sentence Transformers?

**Problem:** Standard, pre-trained Transformer models (like BERT or RoBERTa) are great at language understanding, but they were **not designed to generate good sentence embeddings**.

If you use a vanilla BERT model, you typically get a generic embedding (like the output of the `[CLS]` token). When you compare two of these embeddings, their similarity score is often a **poor predictor of semantic meaning**. Why? The model wasn't trained to cluster similar sentences together in vector space.

**Solution:** **Sentence Transformers (SBERT)**, created by the UKP Lab, is a dedicated Python module that fine-tunes Transformer models specifically for **generating high-quality sentence and paragraph embeddings**. It structures the training process to ensure that semantically similar texts are mapped close together in the vector space.

> 🧠 **Analogy:** If a standard Transformer model is a general-purpose library of knowledge, SBERT is a **highly optimized indexer**. It takes that knowledge and creates an efficient card catalog (the vector space) where cards for similar ideas are placed right next to each other.

**The Three Core Components (Model Types):**

1.  **Sentence Transformer Models (Embedders):**
    * These are used to **compute embeddings** for single sentences or documents.
    * They typically use a **Siamese** or **Triplet Network** structure during training, which forces the distance between similar sentences to be smaller than the distance between dissimilar sentences.
    * The output is a single dense vector (e.g., a 384- or 768-dimensional vector).

2.  **Cross-Encoder Models (Rerankers):**
    * These models **do not** generate individual embeddings.
    * Instead, they take **two sentences** as input simultaneously (separated by a special token) and directly output a single similarity score.
    * They are slower but often achieve **higher accuracy** than Sentence Transformer models for pair-wise tasks because they can perform full cross-attention between the two sentences.

3.  **Sparse Encoder Models:**
    * Used to generate **sparse embeddings**, which are highly effective for certain tasks like **semantic search** and work well with classic Information Retrieval methods.

**Wide Applications:**
By turning text into meaningful, comparable vectors, SBERT unlocks a wide range of powerful applications:

* **Semantic Search:** Finding documents based on meaning, not just keyword overlap.
* **Semantic Textual Similarity (STS):** Accurately scoring how similar two sentences are (the focus of today's practice).
* **Clustering & Grouping:** Automatically organizing large corpuses of text by topic.

 ## Part 2: Setup

 Let's install the libraries we'll need.
 * `pytorch-lightning`: The main library.
 * `sentence-transformers`: For our pre-trained embedding model.
 * `datasets`: To easily load the STSB dataset from Hugging Face.
 * `scipy`: To compute the Spearman correlation (a common metric for this task).

In [1]:
!pip install pytorch-lightning sentence-transformers datasets scipy tensorboard -q

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import pytorch_lightning as pl
from scipy.stats import spearmanr

# Set a seed for reproducibility
pl.seed_everything(42)

/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 42


42

---
## Part 3: The Data

**Task:** We'll use the **Semantic Textual Similarity Benchmark (STSB)**.
* **Input:** Two sentences (e.g., "A man is playing a guitar." and "A man is playing a song.").
* **Label:** A similarity score from 0.0 (no similarity) to 1.0 (perfectly similar).
* **Goal:** Train a model to predict this score.

### 3.1 Load Data
We use the `datasets` library to quickly download it.

In [3]:
# Load the dataset
dataset = load_dataset("sentence-transformers/stsb")


In [4]:
# Let's look at an example
print(dataset['train'][100])

{'sentence1': 'A man is playing a guitar.', 'sentence2': 'Someoen is playing guitar.', 'score': 0.72}


Here you can see that score is 0.72, it's because it's normalized from original human 0-5 scale to 0-1 scale.

### 3.2 Create a PyTorch `Dataset`

# We need to create a custom `Dataset` class to structure our data for PyTorch.

In [5]:
class STSBDataset(Dataset):
    def __init__(self, split='train'):
        # Load the specified split from the correct dataset
        self.data = load_dataset("sentence-transformers/stsb", split=split)
    
    def __len__(self):
        # Return the total number of examples
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get a single example
        item = self.data[idx]
        
        score = item['score']
        
        return {
            'sentence1': item['sentence1'],
            'sentence2': item['sentence2'],
            'label': torch.tensor(score, dtype=torch.float)
        }

 ### 3.3 Create a `collate_fn`

This is a key step! A `DataLoader` batches items together. The default collator tries to stack everything into tensors, but we can't stack *lists of strings*.
Our `collate_fn` will intercept the list of dictionaries from `__getitem__` and organize it into a batch our model can understand:
* A list of `sentence1` strings
* A list of `sentence2` strings
* A tensor of `labels`

In [6]:
def custom_collate_fn(batch):
    """
    batch is a list of dictionaries: [{'sentence1': ..., 'sentence2': ..., 'label': ...}, ...]
    """
    sent1s = [item['sentence1'] for item in batch]
    sent2s = [item['sentence2'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])
    
    return sent1s, sent2s, labels

### 3.4 Create the `DataLoaders`

Now we put it all together. The `DataLoader` will use our `STSBDataset` and `custom_collate_fn` to create batches.

In [7]:
# Create datasets
train_dataset = STSBDataset(split='train')
val_dataset = STSBDataset(split='validation')
test_dataset = STSBDataset(split='test')

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=custom_collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=custom_collate_fn
)

---

## Part 4: The `LightningModule`

This is the heart of our project. We'll define our model architecture and training logic here.

**Our Model Plan:**
1.  Use a pre-trained `SentenceTransformer` ('all-MiniLM-L6-v2') to get embeddings for `sentence1` (`u`) and `sentence2` (`v`).
2.  **Freeze** the `SentenceTransformer`. We're not training it, just using it as a feature extractor.
3.  Create a feature vector: `[u, v, |u - v|]`. This combination is a common and effective trick for similarity tasks.
4.  Pass this feature vector through a simple regression "head" (a few linear layers) to predict a score.

In [8]:
class SentenceSimilarityRegressor(pl.LightningModule):
    
    def __init__(self, model_name='all-MiniLM-L6-v2', learning_rate=1e-3):
        super().__init__()
        
        # Save hyperparameters
        self.save_hyperparameters()
        
        # 1. Load the SentenceTransformer model
        self.sbert = SentenceTransformer(model_name)
        
        # Freeze the SBERT parameters manually
        for param in self.sbert.parameters():
            param.requires_grad = False # This stops them from being updated
        
        # 2. Get the embedding dimension
        sbert_dim = self.sbert.get_sentence_embedding_dimension()
        
        # 3. Create the regression head
        # We concatenate u, v, and |u-v|, so the input dim is sbert_dim * 3
        self.regressor = nn.Sequential(
            nn.Linear(sbert_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid() # To output a score between 0 and 1
        )
        
        # 4. Define the loss function
        self.loss_fn = nn.MSELoss()
        
        # Store validation step outputs
        self.validation_step_outputs = []

        self.test_step_outputs = []

        self.hparams.learning_rate = learning_rate

    def forward(self, sent1_batch, sent2_batch):
        # This is the "inference" part
        
        # 1. Get embeddings
        # We use .encode() which handles tokenization and inference
        # convert_to_tensor=True puts it on the correct device (managed by PL)
        u = self.sbert.encode(sent1_batch, convert_to_tensor=True)
        v = self.sbert.encode(sent2_batch, convert_to_tensor=True)
        
        # 2. Create the feature vector
        diff = torch.abs(u - v)
        features = torch.cat([u, v, diff], dim=1)
        
        # 3. Get prediction
        prediction = self.regressor(features)
        
        # Squeeze to remove extra dimension: [B, 1] -> [B]
        return prediction.squeeze()

    def training_step(self, batch, batch_idx):
        # This is what PL calls for each training batch
        sent1, sent2, labels = batch
        
        # Get predictions
        predictions = self(sent1, sent2)
        
        # Calculate loss
        loss = self.loss_fn(predictions, labels)
        
        # Log the loss
        # 'prog_bar=True' shows it in the progress bar
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss # This is required

    def validation_step(self, batch, batch_idx):
        # This is what PL calls for each validation batch
        sent1, sent2, labels = batch
        
        predictions = self(sent1, sent2)
        loss = self.loss_fn(predictions, labels)
        
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        
        # Store predictions and labels to compute correlation at the end
        # We need to detach them from the graph and move to CPU
        output = {'preds': predictions.detach().cpu(), 'labels': labels.detach().cpu()}
        self.validation_step_outputs.append(output)

    def on_validation_epoch_end(self):
        # This is called at the end of the validation epoch
        # We'll compute the Spearman correlation
        
        all_preds = torch.cat([x['preds'] for x in self.validation_step_outputs])
        all_labels = torch.cat([x['labels'] for x in self.validation_step_outputs])
        
        # Calculate Spearman correlation
        spearman_corr, _ = spearmanr(all_labels.numpy(), all_preds.numpy())
        
        # Log the metric
        self.log('val_spearman', spearman_corr, prog_bar=True)
        
        # Clear the stored outputs
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        # This is what PL calls for each test batch
        sent1, sent2, labels = batch
        
        predictions = self(sent1, sent2)
        loss = self.loss_fn(predictions, labels)
        
        # Log the test loss
        self.log('test_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        
        # Store predictions and labels
        output = {'preds': predictions.detach().cpu(), 'labels': labels.detach().cpu()}
        self.test_step_outputs.append(output)

    def on_test_epoch_end(self):
        # This is called at the end of the test epoch
        all_preds = torch.cat([x['preds'] for x in self.test_step_outputs])
        all_labels = torch.cat([x['labels'] for x in self.test_step_outputs])
        
        # Calculate Spearman correlation
        spearman_corr, _ = spearmanr(all_labels.numpy(), all_preds.numpy())
        
        # Log the final test metric
        self.log('test_spearman', spearman_corr, prog_bar=True)
        
        # Clear the stored outputs
        self.test_step_outputs.clear()

    def configure_optimizers(self):
        # Tell PL what optimizer to use
        # Note: self.parameters() only includes the *trainable* parameters
        # Since self.sbert is frozen, its parameters are not included!
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer

## Part 5: The `Trainer`

Now for the magic. We've defined the *what* (the `LightningModule`), now we just tell the `Trainer` *how* to run it.

In [9]:
# 1. Instantiate our model
model = SentenceSimilarityRegressor()

### 5.1 Quick Test Run

Before full training, it's *always* a good idea to do a "fast development run." This runs 1 batch of training and 1 batch of validation to make sure nothing crashes.

In [10]:
# Create a Trainer
# accelerator='auto' automatically detects if a GPU is available
trainer_test = pl.Trainer(
    fast_dev_run=True,
    accelerator='auto'
)

print("Starting fast_dev_run...")
trainer_test.fit(model, train_loader, val_loader)
print("...fast_dev_run finished successfully!")

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/

Starting fast_dev_run...
Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  3.21it/s, train_loss_step=0.0818, val_loss=0.103, val_spearman=-0.498, train_loss_epoch=0.0818]

`Trainer.fit` stopped: `max_steps=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s, train_loss_step=0.0818, val_loss=0.103, val_spearman=-0.498, train_loss_epoch=0.0818]
...fast_dev_run finished successfully!


 ### 5.2 Full Training

It worked! Now let's train for real. We'll train for 3 epochs. We'll also add `callbacks` for:
* `ModelCheckpoint`: Saves the best model based on `val_spearman`.
* `EarlyStopping`: Stops training if `val_spearman` doesn't improve for 2 epochs.

In [11]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# 1. Define callbacks
checkpoint_callback = ModelCheckpoint(
    monitor='val_spearman', # Metric to watch
    mode='max',             # We want to *maximize* correlation
    filename='best-model-{epoch:02d}-{val_spearman:.3f}',
    save_top_k=1
)

early_stopping_callback = EarlyStopping(
    monitor='val_spearman',
    mode='max',
    patience=2 # Stop if no improvement after 2 epochs
)

# 2. Instantiate our "real" Trainer
trainer = pl.Trainer(
    max_epochs=3,
    accelerator='auto',
    callbacks=[checkpoint_callback, early_stopping_callback],
    # We add a logger (default is TensorBoard)
    logger=pl.loggers.TensorBoardLogger('logs/', name='sentence-similarity')
)

# 3. Start training!
print("Starting full training...")
trainer.fit(model, train_loader, val_loader)
print("...Training complete!")

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type                | Params | Mode 
----------------------------------------------------------
0 | sbert     | SentenceTransformer | 22.7 M | eval 
1 | regressor | Sequential          | 295 K  | train
2 | loss_fn   | MSELoss             | 0      | train
----------------------------------------------------------
295 K     Trainable params
22.7 M    Non-trainable params
23.0 M    Total params
92.035    Total estimated model params size (MB)
7         Modules in train mode
124       Modules in eval mode


Starting full training...
                                                                           

/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:527: Found 124 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 2: 100%|██████████| 180/180 [00:03<00:00, 52.33it/s, v_num=6, train_loss_step=0.0251, val_loss=0.024, val_spearman=0.859, train_loss_epoch=0.0212] 

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|██████████| 180/180 [00:03<00:00, 52.30it/s, v_num=6, train_loss_step=0.0251, val_loss=0.024, val_spearman=0.859, train_loss_epoch=0.0212]
...Training complete!


 Look at that! PL gave us a progress bar, automatic logging, device placement (it used the GPU if available), and checkpointing, all from just defining the `Trainer`.

 ---

## Part 6: Testing & Inference
The `Trainer` object has a `.test()` method to run on the test set. It will automatically load the *best* model checkpoint we saved.

In [12]:
print("Running on test set...")
# trainer.test(dataloaders=test_loader) # This works, but we need to update the model
# For simplicity, let's load the best checkpoint and run test
best_model = SentenceSimilarityRegressor.load_from_checkpoint(checkpoint_callback.best_model_path)

# You can also just call trainer.test() if you didn't reinstantiate the trainer
trainer.test(model=best_model, dataloaders=test_loader)

Running on test set...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/robinhad/Projects/ucu-nlp-course/env/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 44/44 [00:00<00:00, 76.22it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.03031277470290661    │
│       test_spearman       │    0.8191878795623779     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.03031277470290661, 'test_spearman': 0.8191878795623779}]

### 6.1 Inference on New Sentences

Let's use our trained model to predict similarity on new sentences.

In [13]:
# Baseline

# 1. Instantiate a new model instance without loading a checkpoint
initial_model = SentenceSimilarityRegressor().to("cpu")

# 2. Put the initial model in evaluation mode
initial_model.eval()
initial_model.freeze() 

# Define sentence pairs (Same as before)
sent1 = "A cat is sleeping on the couch"
sent2_a = "The feline is napping on the sofa"
sent2_b = "A dog is barking at the mailman"

pair_a_sent1 = [sent1]
pair_a_sent2 = [sent2_a]

pair_b_sent1 = [sent1]
pair_b_sent2 = [sent2_b]

print("--- Initial Model Predictions (Random Head) ---")

with torch.no_grad():
    score_a_init = initial_model(pair_a_sent1, pair_a_sent2)
    score_b_init = initial_model(pair_b_sent1, pair_b_sent2)

print(f"Similarity ('{sent1}' vs '{sent2_a}'): {score_a_init.item():.2f}")
print(f"Similarity ('{sent1}' vs '{sent2_b}'): {score_b_init.item():.2f}")

--- Initial Model Predictions (Random Head) ---
Similarity ('A cat is sleeping on the couch' vs 'The feline is napping on the sofa'): 0.48
Similarity ('A cat is sleeping on the couch' vs 'A dog is barking at the mailman'): 0.48


In [14]:
# Put model in evaluation mode
best_model.eval()
best_model.freeze() # Freeze all weights, including regressor

# Define some sentence pairs
sent1 = "A cat is sleeping on the couch"
sent2_a = "The feline is napping on the sofa"
sent2_b = "A dog is barking at the mailman"

# Model expects a batch (lists of strings)
pair_a_sent1 = [sent1]
pair_a_sent2 = [sent2_a]

pair_b_sent1 = [sent1]
pair_b_sent2 = [sent2_b]

# We need torch.no_grad() for inference
with torch.no_grad():
    score_a = best_model(pair_a_sent1, pair_a_sent2)
    score_b = best_model(pair_b_sent1, pair_b_sent2)

# Remember to convert back from [0, 1] to [0, 5]
print(f"Similarity ('{sent1}' vs '{sent2_a}'): {score_a.item() :.2f}")
print(f"Similarity ('{sent1}' vs '{sent2_b}'): {score_b.item():.2f}")

Similarity ('A cat is sleeping on the couch' vs 'The feline is napping on the sofa'): 0.47
Similarity ('A cat is sleeping on the couch' vs 'A dog is barking at the mailman'): 0.11


 ---

## Part 7: Recap & Next Steps

**What did we do?**
1.  We learned that PyTorch Lightning **organizes** PyTorch code, separating the *science* (`LightningModule`) from the *engineering* (`Trainer`).
2.  We defined our model, loss, and optimization logic in the `LightningModule`.
3.  We defined our data loading using standard PyTorch `Dataset` and `DataLoader`.
4.  We passed *all* of it to the `Trainer`, which handled the loops, GPU, logging, and checkpointing for us.

**Where to go from here?**
* **`LightningDataModule`:** We defined our `DataLoaders` manually. PL has a `LightningDataModule` class that organizes all data-related code (`setup`, `train_dataloader`, `val_dataloader`) in one place.
* **Loggers:** We used the default `TensorBoardLogger`. You can easily swap it for `WandbLogger` (Weights & Biases) or others.
* **Callbacks:** We used `ModelCheckpoint` and `EarlyStopping`. There are many more for things like learning rate scheduling, pruning, and more.
* **Distributed Training:** Changing `accelerator='auto'` and `devices=4` is all you need to do to run on 4 GPUs. PL handles the rest.

### ❓ Q&A